In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")
from IPython.display import display, Javascript

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_districtflowname, get_data, get_data_outbkr, get_homebased_tag

logging.disable(logging.CRITICAL)

In [2]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

In [3]:
def preprocess(df):
    df = get_homebased_tag(df, tag_colname='hb_tag')
    df = get_districtflowname(df, taz_subarea=taz_subarea, taz_colname='otaz', new_colname='o_district')
    df = get_districtflowname(df, taz_subarea=taz_subarea, taz_colname='dtaz', new_colname='d_district')
    return df

# survey
data_fullsurvey['Trip'] = preprocess(data_fullsurvey['Trip'])

# bkrcast
data_daysim['Trip'] = preprocess(data_daysim['Trip'])

In [4]:
summary_survey = data_fullsurvey['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()
summary_daysim = data_daysim['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()

In [5]:
def show_pivot_table(df):
    pivot_survey = (
    df
    .pivot_table(index='o_district', columns='d_district', values='trexpfac', aggfunc='sum', fill_value=0)
    .reindex(index=district_flow_name.values(), columns=district_flow_name.values())
    .fillna(0)
    )

    pivot_survey['Row Total'] = pivot_survey.sum(axis=1)
    pivot_survey.loc['Column Total'] = pivot_survey.sum()
    pivot_survey.columns.name = 'Destination'
    pivot_survey.index.name = 'Origin'

    display(pivot_survey.style.format('{:,.0f}').set_caption('Number of Trips'))
    # show as percent of grand total
    grand_total = pivot_survey.loc['Column Total', 'Row Total']
    if grand_total and grand_total != 0:
        pivot_pct = (pivot_survey / grand_total) * 100
        display(pivot_pct.style.format('{:,.1f}%').set_caption('Trip Share'))

# PSRC Region

## All

In [6]:
show_pivot_table(summary_survey)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"389,595","16,820","9,354","92,572","51,196","13,299","130,987","703,823"
Bellevue Downtown,"20,298","32,881","5,253",144,"1,952","1,497","11,586","73,611"
Kirkland,"13,784","7,213","150,522","31,410","12,889","3,835","72,289","291,942"
Redmond,"75,015","10,558","36,001","155,985","15,957","7,576","90,394","391,486"
Seattle (excluding Seattle downtown),"34,249","3,010","13,385","21,630","2,158,314","193,078","332,702","2,756,368"
Seattle downtown,"16,512","2,628","4,325","7,121","187,158","211,394","79,153","508,290"
Rest,"125,351","10,855","68,226","83,048","353,082","71,159","10,106,807","10,818,527"
Column Total,"674,805","83,964","287,065","391,911","2,780,548","501,837","10,823,917","15,544,048"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),2.5%,0.1%,0.1%,0.6%,0.3%,0.1%,0.8%,4.5%
Bellevue Downtown,0.1%,0.2%,0.0%,0.0%,0.0%,0.0%,0.1%,0.5%
Kirkland,0.1%,0.0%,1.0%,0.2%,0.1%,0.0%,0.5%,1.9%
Redmond,0.5%,0.1%,0.2%,1.0%,0.1%,0.0%,0.6%,2.5%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.1%,0.1%,13.9%,1.2%,2.1%,17.7%
Seattle downtown,0.1%,0.0%,0.0%,0.0%,1.2%,1.4%,0.5%,3.3%
Rest,0.8%,0.1%,0.4%,0.5%,2.3%,0.5%,65.0%,69.6%
Column Total,4.3%,0.5%,1.8%,2.5%,17.9%,3.2%,69.6%,100.0%


In [7]:
show_pivot_table(summary_daysim)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"321,724","48,634","28,681","55,639","35,133","9,559","134,813","634,183"
Bellevue Downtown,"50,117","76,814","11,477","8,004","12,720","1,993","39,020","200,145"
Kirkland,"27,639","11,183","209,581","28,810","18,506","4,290","83,083","383,092"
Redmond,"55,485","8,384","29,392","185,960","12,805","2,631","82,348","377,005"
Seattle (excluding Seattle downtown),"35,287","12,848","18,385","13,474","2,243,324","154,363","358,090","2,835,771"
Seattle downtown,"9,992","2,284","3,959","2,511","155,702","424,822","79,569","678,839"
Rest,"133,914","39,965","81,623","82,579","357,584","81,185","10,348,311","11,125,161"
Column Total,"634,158","200,112","383,098","376,977","2,835,774","678,843","11,125,234","16,234,196"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),2.0%,0.3%,0.2%,0.3%,0.2%,0.1%,0.8%,3.9%
Bellevue Downtown,0.3%,0.5%,0.1%,0.0%,0.1%,0.0%,0.2%,1.2%
Kirkland,0.2%,0.1%,1.3%,0.2%,0.1%,0.0%,0.5%,2.4%
Redmond,0.3%,0.1%,0.2%,1.1%,0.1%,0.0%,0.5%,2.3%
Seattle (excluding Seattle downtown),0.2%,0.1%,0.1%,0.1%,13.8%,1.0%,2.2%,17.5%
Seattle downtown,0.1%,0.0%,0.0%,0.0%,1.0%,2.6%,0.5%,4.2%
Rest,0.8%,0.2%,0.5%,0.5%,2.2%,0.5%,63.7%,68.5%
Column Total,3.9%,1.2%,2.4%,2.3%,17.5%,4.2%,68.5%,100.0%


## HBW

In [8]:
summary_survey_hbw = summary_survey[summary_survey['hb_tag']=='HBW']
show_pivot_table(summary_survey_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"22,409","5,061","2,084","3,382","4,666","2,380","23,426","63,407"
Bellevue Downtown,"4,862",499,136,102,170,372,"7,106","13,247"
Kirkland,"2,077",25,768,"11,198","7,663","2,615","15,859","40,205"
Redmond,"3,151",35,"3,600","13,981","6,246","4,416","16,287","47,717"
Seattle (excluding Seattle downtown),"5,286",260,"7,266","7,926","156,124","40,532","77,017","294,413"
Seattle downtown,"1,613",332,"2,615","3,664","33,067","21,905","29,010","92,206"
Rest,"26,981","7,927","16,877","16,381","83,955","28,832","976,836","1,157,789"
Column Total,"66,378","14,140","33,347","56,634","291,892","101,052","1,145,541","1,708,984"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),1.3%,0.3%,0.1%,0.2%,0.3%,0.1%,1.4%,3.7%
Bellevue Downtown,0.3%,0.0%,0.0%,0.0%,0.0%,0.0%,0.4%,0.8%
Kirkland,0.1%,0.0%,0.0%,0.7%,0.4%,0.2%,0.9%,2.4%
Redmond,0.2%,0.0%,0.2%,0.8%,0.4%,0.3%,1.0%,2.8%
Seattle (excluding Seattle downtown),0.3%,0.0%,0.4%,0.5%,9.1%,2.4%,4.5%,17.2%
Seattle downtown,0.1%,0.0%,0.2%,0.2%,1.9%,1.3%,1.7%,5.4%
Rest,1.6%,0.5%,1.0%,1.0%,4.9%,1.7%,57.2%,67.7%
Column Total,3.9%,0.8%,2.0%,3.3%,17.1%,5.9%,67.0%,100.0%


In [9]:
summary_daysim_hbw = summary_daysim[summary_daysim['hb_tag']=='HBW']
show_pivot_table(summary_daysim_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"9,779","5,664","2,289","5,585","6,975","3,373","18,577","52,242"
Bellevue Downtown,"4,133","2,591","1,731","2,241","4,137",777,"11,856","27,466"
Kirkland,"2,733","2,585","7,684","3,872","3,814","2,439","10,033","33,160"
Redmond,"5,021","3,107","3,121","9,593","4,788","1,327","18,021","44,978"
Seattle (excluding Seattle downtown),"7,620","5,898","3,625","5,850","166,606","53,970","79,942","323,511"
Seattle downtown,"2,537",846,"1,742","1,000","38,366","50,457","28,015","122,963"
Rest,"23,835","18,036","11,592","24,815","93,521","41,770","811,264","1,024,833"
Column Total,"55,658","38,727","31,784","52,956","318,207","154,113","977,708","1,629,153"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.6%,0.3%,0.1%,0.3%,0.4%,0.2%,1.1%,3.2%
Bellevue Downtown,0.3%,0.2%,0.1%,0.1%,0.3%,0.0%,0.7%,1.7%
Kirkland,0.2%,0.2%,0.5%,0.2%,0.2%,0.1%,0.6%,2.0%
Redmond,0.3%,0.2%,0.2%,0.6%,0.3%,0.1%,1.1%,2.8%
Seattle (excluding Seattle downtown),0.5%,0.4%,0.2%,0.4%,10.2%,3.3%,4.9%,19.9%
Seattle downtown,0.2%,0.1%,0.1%,0.1%,2.4%,3.1%,1.7%,7.5%
Rest,1.5%,1.1%,0.7%,1.5%,5.7%,2.6%,49.8%,62.9%
Column Total,3.4%,2.4%,2.0%,3.3%,19.5%,9.5%,60.0%,100.0%


## HBO

In [10]:
summary_survey_hbo = summary_survey[summary_survey['hb_tag']=='HBO']
show_pivot_table(summary_survey_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"144,068","6,808","4,015","20,808","25,331","6,710","57,513","265,254"
Bellevue Downtown,"8,175","5,794","1,630",42,"1,202",791,"2,436","20,069"
Kirkland,"3,822","2,254","64,199","11,877","3,686","1,194","36,327","123,359"
Redmond,"21,255",211,"21,906","76,641","5,483","2,716","37,567","165,778"
Seattle (excluding Seattle downtown),"14,317","1,913","4,452","12,527","1,222,512","74,322","137,449","1,467,492"
Seattle downtown,"12,104",442,"1,621","2,134","75,590","82,012","25,900","199,804"
Rest,"68,565","2,060","30,113","45,778","143,719","24,613","5,077,699","5,392,548"
Column Total,"272,307","19,482","127,937","169,806","1,477,524","192,358","5,374,891","7,634,304"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),1.9%,0.1%,0.1%,0.3%,0.3%,0.1%,0.8%,3.5%
Bellevue Downtown,0.1%,0.1%,0.0%,0.0%,0.0%,0.0%,0.0%,0.3%
Kirkland,0.1%,0.0%,0.8%,0.2%,0.0%,0.0%,0.5%,1.6%
Redmond,0.3%,0.0%,0.3%,1.0%,0.1%,0.0%,0.5%,2.2%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.1%,0.2%,16.0%,1.0%,1.8%,19.2%
Seattle downtown,0.2%,0.0%,0.0%,0.0%,1.0%,1.1%,0.3%,2.6%
Rest,0.9%,0.0%,0.4%,0.6%,1.9%,0.3%,66.5%,70.6%
Column Total,3.6%,0.3%,1.7%,2.2%,19.4%,2.5%,70.4%,100.0%


In [11]:
summary_daysim_hbo = summary_daysim[summary_daysim['hb_tag']=='HBO']
show_pivot_table(summary_daysim_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"172,674","23,722","16,326","28,681","16,634","3,078","78,322","339,437"
Bellevue Downtown,"25,293","31,428","4,273","2,834","3,256",248,"15,342","82,674"
Kirkland,"15,961","4,089","119,657","13,024","8,894",725,"48,447","210,797"
Redmond,"29,195","2,790","13,148","91,590","2,539",360,"42,113","181,735"
Seattle (excluding Seattle downtown),"15,868","2,499","8,576","2,288","1,255,504","55,995","157,055","1,497,785"
Seattle downtown,"3,240",214,739,329,"63,345","181,168","22,236","271,271"
Rest,"73,031","12,918","44,799","36,791","154,299","18,497","6,005,986","6,346,321"
Column Total,"335,262","77,660","207,518","175,537","1,504,471","260,071","6,369,501","8,930,020"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),1.9%,0.3%,0.2%,0.3%,0.2%,0.0%,0.9%,3.8%
Bellevue Downtown,0.3%,0.4%,0.0%,0.0%,0.0%,0.0%,0.2%,0.9%
Kirkland,0.2%,0.0%,1.3%,0.1%,0.1%,0.0%,0.5%,2.4%
Redmond,0.3%,0.0%,0.1%,1.0%,0.0%,0.0%,0.5%,2.0%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.1%,0.0%,14.1%,0.6%,1.8%,16.8%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,0.7%,2.0%,0.2%,3.0%
Rest,0.8%,0.1%,0.5%,0.4%,1.7%,0.2%,67.3%,71.1%
Column Total,3.8%,0.9%,2.3%,2.0%,16.8%,2.9%,71.3%,100.0%


## NHB

In [12]:
summary_survey_nhb = summary_survey[summary_survey['hb_tag']=='NHB']
show_pivot_table(summary_survey_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"193,068","4,628","2,247","68,159","18,869","4,146","34,298","325,415"
Bellevue Downtown,"6,940","26,375","3,487",0,454,334,"2,032","39,622"
Kirkland,"6,878","4,934","74,707","2,692","1,446",26,"19,871","110,555"
Redmond,"47,275","10,311","4,383","59,032","4,228",443,"25,394","151,067"
Seattle (excluding Seattle downtown),"12,426",712,"1,639","1,177","608,642","73,293","108,712","806,601"
Seattle downtown,"2,731","1,853",89,"1,323","74,731","106,994","24,242","211,964"
Rest,"13,735",868,"21,004","8,343","116,861","17,714","3,506,382","3,684,907"
Column Total,"283,053","49,682","107,556","140,727","825,232","202,950","3,720,931","5,330,130"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),3.6%,0.1%,0.0%,1.3%,0.4%,0.1%,0.6%,6.1%
Bellevue Downtown,0.1%,0.5%,0.1%,0.0%,0.0%,0.0%,0.0%,0.7%
Kirkland,0.1%,0.1%,1.4%,0.1%,0.0%,0.0%,0.4%,2.1%
Redmond,0.9%,0.2%,0.1%,1.1%,0.1%,0.0%,0.5%,2.8%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.0%,0.0%,11.4%,1.4%,2.0%,15.1%
Seattle downtown,0.1%,0.0%,0.0%,0.0%,1.4%,2.0%,0.5%,4.0%
Rest,0.3%,0.0%,0.4%,0.2%,2.2%,0.3%,65.8%,69.1%
Column Total,5.3%,0.9%,2.0%,2.6%,15.5%,3.8%,69.8%,100.0%


In [13]:
summary_daysim_nhb = summary_daysim[summary_daysim['hb_tag']=='NHB']
show_pivot_table(summary_daysim_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"111,938","18,163","9,107","19,256","9,965","3,011","29,337","200,777"
Bellevue Downtown,"19,624","42,296","5,393","2,901","4,973",942,"11,485","87,614"
Kirkland,"7,969","4,423","70,083","10,432","4,651","1,090","19,571","118,219"
Redmond,"18,848","2,454","11,486","78,649","4,685",918,"20,127","137,167"
Seattle (excluding Seattle downtown),"10,287","4,137","5,121","4,621","688,369","39,524","101,543","853,602"
Seattle downtown,"4,122","1,205","1,452","1,161","49,340","187,675","27,457","272,412"
Rest,"27,174","8,646","19,870","18,729","88,210","18,838","3,005,073","3,186,540"
Column Total,"199,962","81,324","122,512","135,749","850,193","251,998","3,214,593","4,856,331"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),2.3%,0.4%,0.2%,0.4%,0.2%,0.1%,0.6%,4.1%
Bellevue Downtown,0.4%,0.9%,0.1%,0.1%,0.1%,0.0%,0.2%,1.8%
Kirkland,0.2%,0.1%,1.4%,0.2%,0.1%,0.0%,0.4%,2.4%
Redmond,0.4%,0.1%,0.2%,1.6%,0.1%,0.0%,0.4%,2.8%
Seattle (excluding Seattle downtown),0.2%,0.1%,0.1%,0.1%,14.2%,0.8%,2.1%,17.6%
Seattle downtown,0.1%,0.0%,0.0%,0.0%,1.0%,3.9%,0.6%,5.6%
Rest,0.6%,0.2%,0.4%,0.4%,1.8%,0.4%,61.9%,65.6%
Column Total,4.1%,1.7%,2.5%,2.8%,17.5%,5.2%,66.2%,100.0%


# In-BKR Households

In [14]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)

In [15]:
summary_survey = data_fullsurvey_bkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()
summary_daysim = data_daysim_bkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()

## All

In [16]:
show_pivot_table(summary_survey)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"331,648","15,599","7,771","60,796","28,106","9,333","45,388","498,642"
Bellevue Downtown,"15,438","30,188","5,253",144,772,961,"1,815","54,572"
Kirkland,"9,100","5,412","117,444","29,614","4,300","3,719","24,018","193,607"
Redmond,"44,231","10,558","35,705","129,983","3,922","3,981","45,531","273,912"
Seattle (excluding Seattle downtown),"12,517",768,"4,320","10,170","12,929","17,099","1,333","59,137"
Seattle downtown,"15,607","2,493","3,719","3,560","5,293","6,842","3,275","40,789"
Rest,"47,821",960,"15,766","43,159","1,086","1,653","63,693","174,137"
Column Total,"476,363","65,977","189,979","277,426","56,409","43,589","185,053","1,294,796"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),25.6%,1.2%,0.6%,4.7%,2.2%,0.7%,3.5%,38.5%
Bellevue Downtown,1.2%,2.3%,0.4%,0.0%,0.1%,0.1%,0.1%,4.2%
Kirkland,0.7%,0.4%,9.1%,2.3%,0.3%,0.3%,1.9%,15.0%
Redmond,3.4%,0.8%,2.8%,10.0%,0.3%,0.3%,3.5%,21.2%
Seattle (excluding Seattle downtown),1.0%,0.1%,0.3%,0.8%,1.0%,1.3%,0.1%,4.6%
Seattle downtown,1.2%,0.2%,0.3%,0.3%,0.4%,0.5%,0.3%,3.2%
Rest,3.7%,0.1%,1.2%,3.3%,0.1%,0.1%,4.9%,13.4%
Column Total,36.8%,5.1%,14.7%,21.4%,4.4%,3.4%,14.3%,100.0%


In [17]:
show_pivot_table(summary_daysim)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"283,293","40,863","25,006","49,335","14,616","6,562","27,969","447,644"
Bellevue Downtown,"41,960","59,737","8,946","7,249","2,155",935,"2,479","123,461"
Kirkland,"24,546","8,845","189,861","24,980","11,572","3,828","26,334","289,966"
Redmond,"48,813","7,665","24,719","155,266","5,360","1,936","15,223","258,982"
Seattle (excluding Seattle downtown),"14,485","2,426","11,389","5,231","6,269",509,"1,176","41,485"
Seattle downtown,"6,687","1,173","3,489","1,738",592,"3,972",326,"17,977"
Rest,"27,881","2,733","26,570","15,181",920,235,"13,935","87,455"
Column Total,"447,665","123,442","289,980","258,980","41,484","17,977","87,442","1,266,970"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),22.4%,3.2%,2.0%,3.9%,1.2%,0.5%,2.2%,35.3%
Bellevue Downtown,3.3%,4.7%,0.7%,0.6%,0.2%,0.1%,0.2%,9.7%
Kirkland,1.9%,0.7%,15.0%,2.0%,0.9%,0.3%,2.1%,22.9%
Redmond,3.9%,0.6%,2.0%,12.3%,0.4%,0.2%,1.2%,20.4%
Seattle (excluding Seattle downtown),1.1%,0.2%,0.9%,0.4%,0.5%,0.0%,0.1%,3.3%
Seattle downtown,0.5%,0.1%,0.3%,0.1%,0.0%,0.3%,0.0%,1.4%
Rest,2.2%,0.2%,2.1%,1.2%,0.1%,0.0%,1.1%,6.9%
Column Total,35.3%,9.7%,22.9%,20.4%,3.3%,1.4%,6.9%,100.0%


## HBW

In [18]:
summary_survey_hbw = summary_survey[summary_survey['hb_tag']=='HBW']
show_pivot_table(summary_survey_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"22,409","5,061","2,084","3,382","1,350","2,352","5,255","41,893"
Bellevue Downtown,"4,862",499,136,102,38,278,146,"6,062"
Kirkland,"2,077",25,768,"11,198","2,184","2,569","3,882","22,703"
Redmond,"3,151",35,"3,600","13,981","1,034","2,159","5,702","29,663"
Seattle (excluding Seattle downtown),"1,228",38,"2,184","1,034",0,0,0,"4,484"
Seattle downtown,"1,586",239,"2,569","1,379",0,0,0,"5,772"
Rest,"4,030",101,"3,882","4,286",0,0,0,"12,299"
Column Total,"39,343","5,998","15,223","35,362","4,606","7,359","14,986","122,877"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),18.2%,4.1%,1.7%,2.8%,1.1%,1.9%,4.3%,34.1%
Bellevue Downtown,4.0%,0.4%,0.1%,0.1%,0.0%,0.2%,0.1%,4.9%
Kirkland,1.7%,0.0%,0.6%,9.1%,1.8%,2.1%,3.2%,18.5%
Redmond,2.6%,0.0%,2.9%,11.4%,0.8%,1.8%,4.6%,24.1%
Seattle (excluding Seattle downtown),1.0%,0.0%,1.8%,0.8%,0.0%,0.0%,0.0%,3.6%
Seattle downtown,1.3%,0.2%,2.1%,1.1%,0.0%,0.0%,0.0%,4.7%
Rest,3.3%,0.1%,3.2%,3.5%,0.0%,0.0%,0.0%,10.0%
Column Total,32.0%,4.9%,12.4%,28.8%,3.7%,6.0%,12.2%,100.0%


In [19]:
summary_daysim_hbw = summary_daysim[summary_daysim['hb_tag']=='HBW']
show_pivot_table(summary_daysim_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"9,779","5,664","2,289","5,585","3,511","2,993","5,361","35,182"
Bellevue Downtown,"4,133","2,591","1,731","2,241",434,391,569,"12,090"
Kirkland,"2,733","2,585","7,684","3,872","2,527","2,365","4,683","26,449"
Redmond,"5,021","3,107","3,121","9,593","1,592","1,092","2,953","26,479"
Seattle (excluding Seattle downtown),"2,290",321,"1,733","1,041",0,0,0,"5,385"
Seattle downtown,"1,951",261,"1,610",669,0,0,0,"4,491"
Rest,"3,675",423,"3,227","1,995",0,0,0,"9,320"
Column Total,"29,582","14,952","21,395","24,996","8,064","6,841","13,566","119,396"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),8.2%,4.7%,1.9%,4.7%,2.9%,2.5%,4.5%,29.5%
Bellevue Downtown,3.5%,2.2%,1.4%,1.9%,0.4%,0.3%,0.5%,10.1%
Kirkland,2.3%,2.2%,6.4%,3.2%,2.1%,2.0%,3.9%,22.2%
Redmond,4.2%,2.6%,2.6%,8.0%,1.3%,0.9%,2.5%,22.2%
Seattle (excluding Seattle downtown),1.9%,0.3%,1.5%,0.9%,0.0%,0.0%,0.0%,4.5%
Seattle downtown,1.6%,0.2%,1.3%,0.6%,0.0%,0.0%,0.0%,3.8%
Rest,3.1%,0.4%,2.7%,1.7%,0.0%,0.0%,0.0%,7.8%
Column Total,24.8%,12.5%,17.9%,20.9%,6.8%,5.7%,11.4%,100.0%


## HBO

In [20]:
summary_survey_hbo = summary_survey[summary_survey['hb_tag']=='HBO']
show_pivot_table(summary_survey_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"144,068","6,808","4,015","20,808","20,263","6,595","32,546","235,104"
Bellevue Downtown,"8,175","5,794","1,630",42,473,355,589,"17,057"
Kirkland,"3,822","2,254","64,199","11,877","1,555","1,150","16,611","101,468"
Redmond,"21,255",211,"21,906","76,641","2,137","1,494","13,458","137,103"
Seattle (excluding Seattle downtown),"9,291",389,"2,116","8,632",0,0,0,"20,429"
Seattle downtown,"11,958",401,"1,150",912,0,0,0,"14,420"
Rest,"34,730",691,"8,222","25,364",0,0,115,"69,121"
Column Total,"233,299","16,548","103,238","144,275","24,429","9,593","63,320","594,703"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),24.2%,1.1%,0.7%,3.5%,3.4%,1.1%,5.5%,39.5%
Bellevue Downtown,1.4%,1.0%,0.3%,0.0%,0.1%,0.1%,0.1%,2.9%
Kirkland,0.6%,0.4%,10.8%,2.0%,0.3%,0.2%,2.8%,17.1%
Redmond,3.6%,0.0%,3.7%,12.9%,0.4%,0.3%,2.3%,23.1%
Seattle (excluding Seattle downtown),1.6%,0.1%,0.4%,1.5%,0.0%,0.0%,0.0%,3.4%
Seattle downtown,2.0%,0.1%,0.2%,0.2%,0.0%,0.0%,0.0%,2.4%
Rest,5.8%,0.1%,1.4%,4.3%,0.0%,0.0%,0.0%,11.6%
Column Total,39.2%,2.8%,17.4%,24.3%,4.1%,1.6%,10.6%,100.0%


In [21]:
summary_daysim_hbo = summary_daysim[summary_daysim['hb_tag']=='HBO']
show_pivot_table(summary_daysim_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"172,674","23,722","16,326","28,681","7,034","1,992","15,555","265,984"
Bellevue Downtown,"25,293","31,428","4,273","2,834",445,78,446,"64,797"
Kirkland,"15,961","4,089","119,657","13,024","6,058",685,"14,125","173,599"
Redmond,"29,195","2,790","13,148","91,590","1,742",336,"7,884","146,685"
Seattle (excluding Seattle downtown),"6,644",407,"5,776","1,542",0,0,0,"14,369"
Seattle downtown,"2,107",80,692,304,0,0,0,"3,183"
Rest,"15,258",405,"14,091","7,450",0,0,0,"37,204"
Column Total,"267,132","62,921","173,963","145,425","15,279","3,091","38,010","705,821"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),24.5%,3.4%,2.3%,4.1%,1.0%,0.3%,2.2%,37.7%
Bellevue Downtown,3.6%,4.5%,0.6%,0.4%,0.1%,0.0%,0.1%,9.2%
Kirkland,2.3%,0.6%,17.0%,1.8%,0.9%,0.1%,2.0%,24.6%
Redmond,4.1%,0.4%,1.9%,13.0%,0.2%,0.0%,1.1%,20.8%
Seattle (excluding Seattle downtown),0.9%,0.1%,0.8%,0.2%,0.0%,0.0%,0.0%,2.0%
Seattle downtown,0.3%,0.0%,0.1%,0.0%,0.0%,0.0%,0.0%,0.5%
Rest,2.2%,0.1%,2.0%,1.1%,0.0%,0.0%,0.0%,5.3%
Column Total,37.8%,8.9%,24.6%,20.6%,2.2%,0.4%,5.4%,100.0%


## NHB

In [22]:
summary_survey_nhb = summary_survey[summary_survey['hb_tag']=='NHB']
show_pivot_table(summary_survey_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"135,121","3,408",664,"36,383","5,423",323,"6,819","188,141"
Bellevue Downtown,"2,080","23,682","3,487",0,136,329,"1,067","30,780"
Kirkland,"2,193","3,133","41,630",896,561,0,"3,293","51,707"
Redmond,"16,491","10,311","4,088","33,030",751,328,"15,224","80,223"
Seattle (excluding Seattle downtown),"1,038",215,20,504,"12,929","17,099","1,333","33,139"
Seattle downtown,"2,000","1,853",0,"1,269","5,293","6,842","3,275","20,533"
Rest,"8,295",169,"3,430","1,079","1,086","1,653","63,578","79,290"
Column Total,"167,218","42,771","53,319","73,161","26,179","26,574","94,591","483,812"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),27.9%,0.7%,0.1%,7.5%,1.1%,0.1%,1.4%,38.9%
Bellevue Downtown,0.4%,4.9%,0.7%,0.0%,0.0%,0.1%,0.2%,6.4%
Kirkland,0.5%,0.6%,8.6%,0.2%,0.1%,0.0%,0.7%,10.7%
Redmond,3.4%,2.1%,0.8%,6.8%,0.2%,0.1%,3.1%,16.6%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.0%,0.1%,2.7%,3.5%,0.3%,6.8%
Seattle downtown,0.4%,0.4%,0.0%,0.3%,1.1%,1.4%,0.7%,4.2%
Rest,1.7%,0.0%,0.7%,0.2%,0.2%,0.3%,13.1%,16.4%
Column Total,34.6%,8.8%,11.0%,15.1%,5.4%,5.5%,19.6%,100.0%


In [23]:
summary_daysim_nhb = summary_daysim[summary_daysim['hb_tag']=='NHB']
show_pivot_table(summary_daysim_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"73,507","10,392","5,432","12,952","2,980","1,495","5,868","112,626"
Bellevue Downtown,"11,467","25,219","2,862","2,146",942,443,"1,310","44,389"
Kirkland,"4,876","2,085","50,363","6,602","2,013",746,"5,696","72,381"
Redmond,"12,176","1,735","6,813","47,955","1,273",484,"3,537","73,973"
Seattle (excluding Seattle downtown),"4,621","1,414","3,036","1,982","6,269",509,"1,176","19,007"
Seattle downtown,"2,554",815,"1,166",746,592,"3,972",326,"10,171"
Rest,"7,981","1,772","7,760","5,058",920,235,"13,935","37,661"
Column Total,"117,182","43,432","77,432","77,441","14,989","7,884","31,848","370,208"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),19.9%,2.8%,1.5%,3.5%,0.8%,0.4%,1.6%,30.4%
Bellevue Downtown,3.1%,6.8%,0.8%,0.6%,0.3%,0.1%,0.4%,12.0%
Kirkland,1.3%,0.6%,13.6%,1.8%,0.5%,0.2%,1.5%,19.6%
Redmond,3.3%,0.5%,1.8%,13.0%,0.3%,0.1%,1.0%,20.0%
Seattle (excluding Seattle downtown),1.2%,0.4%,0.8%,0.5%,1.7%,0.1%,0.3%,5.1%
Seattle downtown,0.7%,0.2%,0.3%,0.2%,0.2%,1.1%,0.1%,2.7%
Rest,2.2%,0.5%,2.1%,1.4%,0.2%,0.1%,3.8%,10.2%
Column Total,31.7%,11.7%,20.9%,20.9%,4.0%,2.1%,8.6%,100.0%


# Outside-BKR Households

In [24]:
data_daysim_outbkr, data_survey_outbkr, data_fullsurvey_outbkr = \
    get_data_outbkr(data1=data_daysim, data2=data_survey, data3=data_fullsurvey, taz_subarea=taz_subarea)

In [25]:
summary_survey = data_fullsurvey_outbkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()
summary_daysim = data_daysim_outbkr['Trip'].groupby(by=['o_district', 'd_district', 'dpurp', 'hb_tag'])['trexpfac'].sum().reset_index()

## All

In [26]:
show_pivot_table(summary_survey)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"57,947","1,221","1,583","31,776","23,090","3,965","85,600","205,181"
Bellevue Downtown,"4,860","2,693",0,0,"1,180",536,"9,770","19,039"
Kirkland,"4,685","1,801","33,077","1,796","8,589",116,"48,271","98,335"
Redmond,"30,784",0,295,"26,002","12,034","3,595","44,863","117,574"
Seattle (excluding Seattle downtown),"21,732","2,242","9,065","11,460","2,145,385","175,979","331,368","2,697,231"
Seattle downtown,905,135,606,"3,561","181,865","204,552","75,878","467,501"
Rest,"77,530","9,895","52,460","39,889","351,996","69,506","10,043,114","10,644,390"
Column Total,"198,443","17,987","97,087","114,484","2,724,139","458,248","10,638,864","14,249,252"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.4%,0.0%,0.0%,0.2%,0.2%,0.0%,0.6%,1.4%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.1%,0.1%
Kirkland,0.0%,0.0%,0.2%,0.0%,0.1%,0.0%,0.3%,0.7%
Redmond,0.2%,0.0%,0.0%,0.2%,0.1%,0.0%,0.3%,0.8%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.1%,0.1%,15.1%,1.2%,2.3%,18.9%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.3%,1.4%,0.5%,3.3%
Rest,0.5%,0.1%,0.4%,0.3%,2.5%,0.5%,70.5%,74.7%
Column Total,1.4%,0.1%,0.7%,0.8%,19.1%,3.2%,74.7%,100.0%


In [27]:
show_pivot_table(summary_daysim)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"38,431","7,771","3,675","6,304","20,517","2,997","106,844","186,539"
Bellevue Downtown,"8,157","17,077","2,531",755,"10,565","1,058","36,541","76,684"
Kirkland,"3,093","2,338","19,720","3,830","6,934",462,"56,749","93,126"
Redmond,"6,672",719,"4,673","30,694","7,445",695,"67,125","118,023"
Seattle (excluding Seattle downtown),"20,802","10,422","6,996","8,243","2,237,055","153,854","356,914","2,794,286"
Seattle downtown,"3,305","1,111",470,773,"155,110","420,850","79,243","660,862"
Rest,"106,033","37,232","55,053","67,398","356,664","80,950","10,334,376","11,037,706"
Column Total,"186,493","76,670","93,118","117,997","2,794,290","660,866","11,037,792","14,967,226"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.3%,0.1%,0.0%,0.0%,0.1%,0.0%,0.7%,1.2%
Bellevue Downtown,0.1%,0.1%,0.0%,0.0%,0.1%,0.0%,0.2%,0.5%
Kirkland,0.0%,0.0%,0.1%,0.0%,0.0%,0.0%,0.4%,0.6%
Redmond,0.0%,0.0%,0.0%,0.2%,0.0%,0.0%,0.4%,0.8%
Seattle (excluding Seattle downtown),0.1%,0.1%,0.0%,0.1%,14.9%,1.0%,2.4%,18.7%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.0%,2.8%,0.5%,4.4%
Rest,0.7%,0.2%,0.4%,0.5%,2.4%,0.5%,69.0%,73.7%
Column Total,1.2%,0.5%,0.6%,0.8%,18.7%,4.4%,73.7%,100.0%


## HBW

In [28]:
summary_survey_hbw = summary_survey[summary_survey['hb_tag']=='HBW']
show_pivot_table(summary_survey_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0,0,0,0,"3,316",28,"18,170","21,514"
Bellevue Downtown,0,0,0,0,133,93,"6,960","7,186"
Kirkland,0,0,0,0,"5,479",46,"11,977","17,502"
Redmond,0,0,0,0,"5,213","2,257","10,585","18,055"
Seattle (excluding Seattle downtown),"4,058",222,"5,082","6,892","156,124","40,532","77,017","289,929"
Seattle downtown,28,93,46,"2,285","33,067","21,905","29,010","86,433"
Rest,"22,950","7,826","12,995","12,095","83,955","28,832","976,836","1,145,490"
Column Total,"27,035","8,142","18,123","21,272","287,286","93,693","1,130,556","1,586,107"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.0%,0.0%,0.0%,0.0%,0.2%,0.0%,1.1%,1.4%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.4%,0.5%
Kirkland,0.0%,0.0%,0.0%,0.0%,0.3%,0.0%,0.8%,1.1%
Redmond,0.0%,0.0%,0.0%,0.0%,0.3%,0.1%,0.7%,1.1%
Seattle (excluding Seattle downtown),0.3%,0.0%,0.3%,0.4%,9.8%,2.6%,4.9%,18.3%
Seattle downtown,0.0%,0.0%,0.0%,0.1%,2.1%,1.4%,1.8%,5.4%
Rest,1.4%,0.5%,0.8%,0.8%,5.3%,1.8%,61.6%,72.2%
Column Total,1.7%,0.5%,1.1%,1.3%,18.1%,5.9%,71.3%,100.0%


In [29]:
summary_daysim_hbw = summary_daysim[summary_daysim['hb_tag']=='HBW']
show_pivot_table(summary_daysim_hbw)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0,0,0,0,"3,464",380,"13,216","17,060"
Bellevue Downtown,0,0,0,0,"3,703",386,"11,287","15,376"
Kirkland,0,0,0,0,"1,287",74,"5,350","6,711"
Redmond,0,0,0,0,"3,196",235,"15,068","18,499"
Seattle (excluding Seattle downtown),"5,330","5,577","1,892","4,809","166,606","53,970","79,942","318,126"
Seattle downtown,586,585,132,331,"38,366","50,457","28,015","118,472"
Rest,"20,160","17,613","8,365","22,820","93,521","41,770","811,264","1,015,513"
Column Total,"26,076","23,775","10,389","27,960","310,143","147,272","964,142","1,509,757"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.0%,0.0%,0.0%,0.0%,0.2%,0.0%,0.9%,1.1%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.2%,0.0%,0.7%,1.0%
Kirkland,0.0%,0.0%,0.0%,0.0%,0.1%,0.0%,0.4%,0.4%
Redmond,0.0%,0.0%,0.0%,0.0%,0.2%,0.0%,1.0%,1.2%
Seattle (excluding Seattle downtown),0.4%,0.4%,0.1%,0.3%,11.0%,3.6%,5.3%,21.1%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,2.5%,3.3%,1.9%,7.8%
Rest,1.3%,1.2%,0.6%,1.5%,6.2%,2.8%,53.7%,67.3%
Column Total,1.7%,1.6%,0.7%,1.9%,20.5%,9.8%,63.9%,100.0%


## HBO

In [30]:
summary_survey_hbo = summary_survey[summary_survey['hb_tag']=='HBO']
show_pivot_table(summary_survey_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0,0,0,0,"5,068",115,"24,967","30,150"
Bellevue Downtown,0,0,0,0,729,437,"1,846","3,012"
Kirkland,0,0,0,0,"2,131",44,"19,716","21,891"
Redmond,0,0,0,0,"3,345","1,222","24,108","28,676"
Seattle (excluding Seattle downtown),"5,026","1,523","2,336","3,894","1,222,512","74,322","137,449","1,447,062"
Seattle downtown,147,42,471,"1,222","75,590","82,012","25,900","185,384"
Rest,"33,835","1,369","21,891","20,414","143,719","24,613","5,077,584","5,323,426"
Column Total,"39,008","2,934","24,699","25,531","1,453,095","182,765","5,311,571","7,039,602"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.0%,0.0%,0.0%,0.0%,0.1%,0.0%,0.4%,0.4%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
Kirkland,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.3%,0.3%
Redmond,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.3%,0.4%
Seattle (excluding Seattle downtown),0.1%,0.0%,0.0%,0.1%,17.4%,1.1%,2.0%,20.6%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.1%,1.2%,0.4%,2.6%
Rest,0.5%,0.0%,0.3%,0.3%,2.0%,0.3%,72.1%,75.6%
Column Total,0.6%,0.0%,0.4%,0.4%,20.6%,2.6%,75.5%,100.0%


In [31]:
summary_daysim_hbo = summary_daysim[summary_daysim['hb_tag']=='HBO']
show_pivot_table(summary_daysim_hbo)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0,0,0,0,"9,600","1,086","62,767","73,453"
Bellevue Downtown,0,0,0,0,"2,811",170,"14,896","17,877"
Kirkland,0,0,0,0,"2,836",40,"34,322","37,198"
Redmond,0,0,0,0,797,24,"34,229","35,050"
Seattle (excluding Seattle downtown),"9,224","2,092","2,800",746,"1,255,504","55,995","157,055","1,483,416"
Seattle downtown,"1,133",134,47,25,"63,345","181,168","22,236","268,088"
Rest,"57,773","12,513","30,708","29,341","154,299","18,497","6,005,986","6,309,117"
Column Total,"68,130","14,739","33,555","30,112","1,489,192","256,980","6,331,491","8,224,199"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.0%,0.0%,0.0%,0.0%,0.1%,0.0%,0.8%,0.9%
Bellevue Downtown,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.2%,0.2%
Kirkland,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.4%,0.5%
Redmond,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%,0.4%,0.4%
Seattle (excluding Seattle downtown),0.1%,0.0%,0.0%,0.0%,15.3%,0.7%,1.9%,18.0%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,0.8%,2.2%,0.3%,3.3%
Rest,0.7%,0.2%,0.4%,0.4%,1.9%,0.2%,73.0%,76.7%
Column Total,0.8%,0.2%,0.4%,0.4%,18.1%,3.1%,77.0%,100.0%


## NHB

In [32]:
summary_survey_nhb = summary_survey[summary_survey['hb_tag']=='NHB']
show_pivot_table(summary_survey_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"57,947","1,221","1,583","31,776","13,445","3,823","27,479","137,274"
Bellevue Downtown,"4,860","2,693",0,0,318,5,965,"8,842"
Kirkland,"4,685","1,801","33,077","1,796",885,26,"16,577","58,848"
Redmond,"30,784",0,295,"26,002","3,476",115,"10,170","70,844"
Seattle (excluding Seattle downtown),"11,388",497,"1,619",673,"595,713","56,193","107,379","773,461"
Seattle downtown,731,0,89,54,"69,438","100,152","20,967","191,431"
Rest,"5,440",700,"17,574","7,264","115,776","16,061","3,442,803","3,605,617"
Column Total,"115,835","6,911","54,237","67,566","799,052","176,376","3,626,340","4,846,317"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),1.2%,0.0%,0.0%,0.7%,0.3%,0.1%,0.6%,2.8%
Bellevue Downtown,0.1%,0.1%,0.0%,0.0%,0.0%,0.0%,0.0%,0.2%
Kirkland,0.1%,0.0%,0.7%,0.0%,0.0%,0.0%,0.3%,1.2%
Redmond,0.6%,0.0%,0.0%,0.5%,0.1%,0.0%,0.2%,1.5%
Seattle (excluding Seattle downtown),0.2%,0.0%,0.0%,0.0%,12.3%,1.2%,2.2%,16.0%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.4%,2.1%,0.4%,4.0%
Rest,0.1%,0.0%,0.4%,0.1%,2.4%,0.3%,71.0%,74.4%
Column Total,2.4%,0.1%,1.1%,1.4%,16.5%,3.6%,74.8%,100.0%


In [33]:
summary_daysim_nhb = summary_daysim[summary_daysim['hb_tag']=='NHB']
show_pivot_table(summary_daysim_nhb)

Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),"38,431","7,771","3,675","6,304","6,985","1,516","23,469","88,151"
Bellevue Downtown,"8,157","17,077","2,531",755,"4,031",499,"10,175","43,225"
Kirkland,"3,093","2,338","19,720","3,830","2,638",344,"13,875","45,838"
Redmond,"6,672",719,"4,673","30,694","3,412",434,"16,590","63,194"
Seattle (excluding Seattle downtown),"5,666","2,723","2,085","2,639","682,100","39,015","100,367","834,595"
Seattle downtown,"1,568",390,286,415,"48,748","183,703","27,131","262,241"
Rest,"19,193","6,874","12,110","13,671","87,290","18,603","2,991,138","3,148,879"
Column Total,"82,780","37,892","45,080","58,308","835,204","244,114","3,182,745","4,486,123"


Destination,Bellevue (excluding downtown),Bellevue Downtown,Kirkland,Redmond,Seattle (excluding Seattle downtown),Seattle downtown,Rest,Row Total
Origin,,,,,,,,
Bellevue (excluding downtown),0.9%,0.2%,0.1%,0.1%,0.2%,0.0%,0.5%,2.0%
Bellevue Downtown,0.2%,0.4%,0.1%,0.0%,0.1%,0.0%,0.2%,1.0%
Kirkland,0.1%,0.1%,0.4%,0.1%,0.1%,0.0%,0.3%,1.0%
Redmond,0.1%,0.0%,0.1%,0.7%,0.1%,0.0%,0.4%,1.4%
Seattle (excluding Seattle downtown),0.1%,0.1%,0.0%,0.1%,15.2%,0.9%,2.2%,18.6%
Seattle downtown,0.0%,0.0%,0.0%,0.0%,1.1%,4.1%,0.6%,5.8%
Rest,0.4%,0.2%,0.3%,0.3%,1.9%,0.4%,66.7%,70.2%
Column Total,1.8%,0.8%,1.0%,1.3%,18.6%,5.4%,70.9%,100.0%
